In [1]:
# CÉLULA 1 — Instalação das bibliotecas 
!pip install requests tqdm plotly seaborn matplotlib streamlit pandas numpy scipy statsmodels

In [2]:
# CÉLULA 2: Imports e configuração 

import os
import zipfile
import requests
from tqdm.notebook import tqdm

# ── Diretório base do projeto ──
BASE_DIR = r"C:\Users\Usuario\Downloads\Bruno_USP\Documentos\Censo_Religião"
DATA_DIR = os.path.join(BASE_DIR, "data", "raw")

# ── Cria as pastas se ainda não existirem ──
os.makedirs(DATA_DIR, exist_ok=True)

print("Configuração carregada!")
print(f"Dados serão salvos em: {DATA_DIR}")

Configuração carregada!
Dados serão salvos em: C:\Users\Usuario\Downloads\Bruno_USP\Documentos\Censo_Religião\data\raw


In [3]:
# CÉLULA 3: Mapeamento dos arquivos no FTP do IBGE 

BASE_FTP = "https://ftp.ibge.gov.br/Censos"

SIGLAS_UF = ["AC", "AL", "AM", "AP", "BA", "CE", "DF", "ES", "GO", 
             "MA", "MG", "MS", "MT", "PA", "PB", "PE", "PI", "PR",
             "RJ", "RN", "RO", "RR", "RS", "SC", "SE", "SP", "TO"]

# URLs do censo 1991 — arquivo único nacional
urls_1991 = [f"{BASE_FTP}/Censo_Demografico_1991/Microdados/Microdados_Censo_Demografico_1991_Amostra.zip"]

# URLs do censo 2000 — um arquivo por estado
urls_2000 = ([f"{BASE_FTP}/Censo_Demografico_2000/Microdados/1_Documentacao_20170908.zip"] +
    [f"{BASE_FTP}/Censo_Demografico_2000/Microdados/{uf}.zip" for uf in SIGLAS_UF])

# URLs do censo 2010 — um arquivo por estado (SP dividido em dois)
urls_2010 = ([f"{BASE_FTP}/Censo_Demografico_2010/Resultados_Gerais_da_Amostra/Microdados/Documentacao.zip"] +
    [f"{BASE_FTP}/Censo_Demografico_2010/Resultados_Gerais_da_Amostra/Microdados/{uf}.zip"
     for uf in SIGLAS_UF if uf != "SP"] + [f"{BASE_FTP}/Censo_Demografico_2010/Resultados_Gerais_da_Amostra/Microdados/SP1.zip",
     f"{BASE_FTP}/Censo_Demografico_2010/Resultados_Gerais_da_Amostra/Microdados/SP2_RM.zip"])

# Resumo
print(f"1991: {len(urls_1991)} arquivo(s)")
print(f"2000: {len(urls_2000)} arquivo(s)")
print(f"2010: {len(urls_2010)} arquivo(s)")

1991: 1 arquivo(s)
2000: 28 arquivo(s)
2010: 29 arquivo(s)


In [9]:
# CÉLULA 4 — Funções de download e extração

def instalar_7zip():
    import subprocess
    print("Instalando 7-zip...")
    subprocess.run(["winget", "install", "--id", "7zip.7zip", "-e", "--silent"])
    print("7-zip instalado!")

def baixar_arquivo(url, pasta):
    nome = url.split("/")[-1]
    caminho = os.path.join(pasta, nome)
    if os.path.isfile(caminho):
        print(f"Já existe, pulando: {nome}")
        return
    print(f"Baixando: {nome}")
    resposta = requests.get(url, stream=True)
    tamanho = int(resposta.headers.get("content-length", 0))
    with open(caminho, "wb") as f, tqdm(total=tamanho, unit="iB", unit_scale=True) as barra:
        for bloco in resposta.iter_content(chunk_size=8192):
            f.write(bloco)
            barra.update(len(bloco))
    print(f"Concluído: {nome}")

def descompactar(caminho_zip, pasta):
    import subprocess
    sete_zip = r"C:\Program Files (x86)\7-Zip\7z.exe"
    if not os.path.isfile(sete_zip):
        instalar_7zip()
    subprocess.run([sete_zip, "x", caminho_zip, f"-o{pasta}", "-y"])
    print(f"Extraído: {os.path.basename(caminho_zip)}")

print("Funções carregadas!")

Funções carregadas!


In [5]:
# CÉLULA 5 — Download dos microdados

pasta_1991 = os.path.join(DATA_DIR, "1991")
pasta_2000 = os.path.join(DATA_DIR, "2000")
pasta_2010 = os.path.join(DATA_DIR, "2010")

os.makedirs(pasta_1991, exist_ok=True)
os.makedirs(pasta_2000, exist_ok=True)
os.makedirs(pasta_2010, exist_ok=True)

print("Baixando 1991...")
for url in urls_1991:
    baixar_arquivo(url, pasta_1991)

print("Baixando 2000...")
for url in urls_2000:
    baixar_arquivo(url, pasta_2000)

print("Baixando 2010...")
for url in urls_2010:
    baixar_arquivo(url, pasta_2010)

print("Download completo!")

Baixando 1991...
Já existe, pulando: Microdados_Censo_Demografico_1991_Amostra.zip
Baixando 2000...
Já existe, pulando: 1_Documentacao_20170908.zip
Já existe, pulando: AC.zip
Já existe, pulando: AL.zip
Já existe, pulando: AM.zip
Já existe, pulando: AP.zip
Já existe, pulando: BA.zip
Já existe, pulando: CE.zip
Já existe, pulando: DF.zip
Já existe, pulando: ES.zip
Já existe, pulando: GO.zip
Já existe, pulando: MA.zip
Já existe, pulando: MG.zip
Já existe, pulando: MS.zip
Já existe, pulando: MT.zip
Já existe, pulando: PA.zip
Já existe, pulando: PB.zip
Já existe, pulando: PE.zip
Já existe, pulando: PI.zip
Já existe, pulando: PR.zip
Já existe, pulando: RJ.zip
Já existe, pulando: RN.zip
Já existe, pulando: RO.zip
Já existe, pulando: RR.zip
Já existe, pulando: RS.zip
Já existe, pulando: SC.zip
Já existe, pulando: SE.zip
Já existe, pulando: SP.zip
Já existe, pulando: TO.zip
Baixando 2010...
Já existe, pulando: Documentacao.zip
Já existe, pulando: AC.zip
Já existe, pulando: AL.zip
Já existe, pula

In [8]:
# CÉLULA 6 — Extração dos arquivos

import os

def extrair_censo(pasta):
    for arquivo in os.listdir(pasta):
        if arquivo.endswith(".zip"):
            caminho_zip = os.path.join(pasta, arquivo)
            pasta_extracao = os.path.join(pasta, arquivo.replace(".zip", ""))
            os.makedirs(pasta_extracao, exist_ok=True)
            descompactar(caminho_zip, pasta_extracao)

print("Extraindo 1991...")
extrair_censo(pasta_1991)

print("Extraindo 2000...")
extrair_censo(pasta_2000)

print("Extraindo 2010...")
extrair_censo(pasta_2010)

print("Extração completa!")

Extraindo 1991...
Extraído: Microdados_Censo_Demografico_1991_Amostra.zip
Extraindo 2000...
Extraído: 1_Documentacao_20170908.zip
Extraído: AC.zip
Extraído: AL.zip
Extraído: AM.zip
Extraído: AP.zip
Extraído: BA.zip
Extraído: CE.zip
Extraído: DF.zip
Extraído: ES.zip
Extraído: GO.zip
Extraído: MA.zip
Extraído: MG.zip
Extraído: MS.zip
Extraído: MT.zip
Extraído: PA.zip
Extraído: PB.zip
Extraído: PE.zip
Extraído: PI.zip
Extraído: PR.zip
Extraído: RJ.zip
Extraído: RN.zip
Extraído: RO.zip
Extraído: RR.zip
Extraído: RS.zip
Extraído: SC.zip
Extraído: SE.zip
Extraído: SP.zip
Extraído: TO.zip
Extraindo 2010...
Extraído: AC.zip
Extraído: AL.zip
Extraído: AM.zip
Extraído: AP.zip
Extraído: BA.zip
Extraído: CE.zip
Extraído: DF.zip
Extraído: Documentacao.zip
Extraído: ES.zip
Extraído: GO.zip
Extraído: MA.zip
Extraído: MG.zip
Extraído: MS.zip
Extraído: MT.zip
Extraído: PA.zip
Extraído: PB.zip
Extraído: PE.zip
Extraído: PI.zip
Extraído: PR.zip
Extraído: RJ.zip
Extraído: RN.zip
Extraído: RO.zip
Extraído: